In [1]:
# This script extracts AND processes acoustic features from waveform audio files
# All lines where "TRF for Alice EEG Dataset" pipeline (cited in the thesis) is followed are commented with: "TRF for Alice EEG Dataset"

from pathlib import Path
import numpy as np
import librosa
from scipy.signal import butter, filtfilt
import eelbrain
import pickle

# Define paths that will be used throughout
DATA_ROOT = Path('/Users/zorkabozilovic/Desktop/PART1')
STIMULUS_DIR = DATA_ROOT / 'diliBach_wav_4dryad'

# Audio feature extraction parameters
AUDIO_SR = 44100
TARGET_SR = 100

In [2]:
# Extract acoustic features from WAV files

hop_length = AUDIO_SR // TARGET_SR  # 441 (exactly 100 frames/sec)
song_features = {}

for song_id in range(1, 11):
    
    wav_path = STIMULUS_DIR / f'{song_id}.wav'
    print(f'Song {song_id}/10')
   
    # Envelope (following "TRF for Alice EEG Dataset")
    wav_ndvar = eelbrain.load.wav(wav_path)
    envelope = wav_ndvar.envelope()
    envelope = eelbrain.resample(envelope, TARGET_SR)
    
    # Onsets
    onset = envelope.diff('time').clip(0) # following "TRF for Alice EEG Dataset"
    # Low-pass filter onsets to keep only prominent peaks
    b, a = butter(N=2, Wn = 10 / (TARGET_SR / 2), btype='low')
    onset_filt = filtfilt(b, a, onset.x)
    onset = eelbrain.NDVar(np.clip(onset_filt, 0, None), (envelope.time,))
    
    # Load with librosa
    y, sr = librosa.load(str(wav_path), sr=AUDIO_SR, mono=True)
   
    # Pitch (F0) with pyin
    f0, voiced_flag, voiced_prob = librosa.pyin (  # only f0 is used here
        y, sr=sr,
        fmin=librosa.note_to_hz('E3'), # set to 3 semitones below lowest note (G3) as safety margin
        fmax=librosa.note_to_hz('C7'), # set to 3 semitones above highest note (A6) as safety margin
        hop_length=hop_length,
    )
    # Interpolate through unvoiced NaN frames
    nan_mask = np.isnan(f0)
    if nan_mask.all():
        f0 = np.zeros_like(f0)
    elif nan_mask.any():
        voiced_indices = np.where(~nan_mask)[0]
        f0 = np.interp(np.arange(len(f0)), voiced_indices, f0[voiced_indices])
    # Compute pitch as spikes at every note change
    pitch_for_change = eelbrain.NDVar(f0, (eelbrain.UTS(0, 1/TARGET_SR, len(f0)),))
    pitch_note_change = pitch_for_change.diff('time').abs()
   
    # Spectral centroid
    cent = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop_length)[0]
    
    # MFCC 3
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=4, n_fft=2048, hop_length=hop_length, fmax=8000)
    mfcc3 = mfccs[3] # 3 is spectral fine structure

    # Trim librosa features to match envelope length
    n = len(envelope)
    pitch_change = pitch_note_change.x[:n]
    cent = cent[:n]
    mfcc3 = mfcc3[:n]

    # Wrap in NDVars using envelope's time axis
    pitch_ndvar = eelbrain.NDVar(pitch_change, (envelope.time,))
    cent_ndvar = eelbrain.NDVar(cent, (envelope.time,))
    mfcc3_ndvar = eelbrain.NDVar(mfcc3, (envelope.time,))  

    # Save all features
    song_features[song_id] = {
        'envelope': envelope,
        'onsets': onset,
        'pitch': pitch_ndvar,
        'centroid': cent_ndvar,
        'mfcc3': mfcc3_ndvar,
    }

Song 1/10
Song 2/10
Song 3/10
Song 4/10
Song 5/10
Song 6/10
Song 7/10
Song 8/10
Song 9/10
Song 10/10


In [3]:
# Standardisation for all features for all 10 songs

for feat_name in ['envelope', 'onsets', 'pitch', 'centroid', 'mfcc3']:
    all_vals = np.concatenate([song_features[i][feat_name].x for i in range(1, 11)])
    feat_mean = all_vals.mean()
    feat_std = all_vals.std()
    for song_id in range(1, 11):
        old = song_features[song_id][feat_name]
        song_features[song_id][feat_name] = eelbrain.NDVar((old.x - feat_mean) / (feat_std + 1e-10), old.dims)

In [4]:
# Save all features

out_path = DATA_ROOT / 'improved_song_features.pkl'
with open(out_path, 'wb') as f:
    pickle.dump(song_features, f)
print('Features saved')

Features saved
